# 🇳🇬 Currency Crisis Early Warning Model — Nigerian Economy
## Random Forest Ensemble Classifier  |  1990–2023

**Author:** [Your Name]  
**Algorithm:** Random Forest (Bootstrap Aggregating Ensemble)  
**Target:** Binary Currency Crisis (1 = Crisis, 0 = No Crisis)  
**Crisis Definition:** EMP index > μ + 1.5σ  OR  |ΔE/E| > 15%  OR  ΔR/R < −15%

---
### Notebook Sections
1. 📦 Install & Import Libraries  
2. 📊 Data Loading & Inspection  
3. 🔍 Exploratory Data Analysis (EDA)  
4. 📐 EMP Index Construction & Target Label  
5. ⚙️ Feature Engineering & Preprocessing  
6. 🌲 Random Forest Training & Hyperparameter Tuning  
7. 📈 Model Evaluation (AUC-ROC, F1, OOB Score)  
8. 🔎 SHAP Explainability & Feature Importance  
9. 🔄 Walk-Forward Cross-Validation  
10. 🚨 Early Warning Score & Policy Signals  
11. 📋 Summary Report  


## 1. Install & Import Libraries

In [ ]:
# Install required packages (run once in Colab)
!pip install shap imbalanced-learn optuna --quiet


In [ ]:
# ── Core ─────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (TimeSeriesSplit, cross_val_score,
                                     GridSearchCV, StratifiedKFold)
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                              recall_score, accuracy_score, confusion_matrix,
                              RocCurveDisplay, classification_report,
                              brier_score_loss, precision_recall_curve)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance

# ── Imbalanced learning ───────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# ── SHAP ──────────────────────────────────────────────────────────────────────
import shap

# ── Hyperparameter search ─────────────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})
NG_GREEN = '#008751'
NG_GOLD  = '#FFD700'
CRISIS_RED = '#C62828'

print("✅  All libraries loaded successfully")


## 2. Data Loading & Inspection

In [ ]:
# ── Nigeria Currency Crisis Dataset (1990–2023) ───────────────────────────────
# Upload 'Nigeria_Currency_Crisis_Dataset.xlsx' OR use the inline data below.

raw = {
    'Year':             [1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,
                         2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,
                         2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,
                         2020,2021,2022,2023],
    'ExRate_NGBUSD':    [8.0,9.9,17.3,22.1,21.9,81.0,82.0,82.0,84.0,92.7,
                         101.7,111.2,120.6,129.2,133.5,132.1,128.3,125.8,118.6,148.9,
                         150.3,154.7,157.3,157.3,158.6,196.5,305.0,330.0,306.0,306.9,
                         379.0,411.0,446.0,900.0],
    'ExRate_Depr_pct':  [-3.5,23.8,74.7,27.7,-0.9,269.9,1.2,0.0,2.4,10.4,
                          9.7,9.3,8.4,7.1,3.3,-1.0,-2.9,-1.9,-5.7,25.5,
                          0.9,2.9,1.7,0.0,0.8,23.9,55.2,8.2,-7.3,0.3,
                          23.5,8.4,8.5,101.8],
    'Parallel_Premium': [2.5,5.1,12.3,68.5,312.8,0.8,2.5,3.2,5.8,18.5,
                          8.2,5.5,8.3,6.2,4.5,3.2,2.8,2.2,3.5,8.5,
                          3.8,3.2,2.8,3.5,4.2,32.5,58.8,38.5,8.5,5.5,
                          25.8,55.8,118.5,35.2],
    'FX_Reserves_USDbn':[3.8,3.5,1.0,1.5,1.7,1.4,4.1,7.6,7.1,5.4,
                          9.9,8.2,7.3,10.4,17.0,28.3,42.3,51.5,53.0,42.4,
                          32.3,35.2,43.8,42.8,34.5,28.3,24.8,38.8,43.0,38.0,
                          35.6,40.5,36.6,33.5],
    'Reserves_Change_pct':[8.2,-7.9,-71.4,50.0,13.3,-17.6,192.9,85.4,-6.6,-24.0,
                            83.3,-17.2,-11.0,42.5,63.5,66.5,49.5,21.8,2.9,-20.0,
                            -23.8,9.0,24.4,-2.3,-19.4,-18.0,-12.4,56.5,10.8,-11.6,
                            -6.3,13.8,-9.6,-8.5],
    'EMP_Index':        [-0.3,1.2,5.8,3.5,-1.5,8.9,-1.2,-0.8,0.5,2.8,
                          1.5,1.8,1.9,1.2,0.5,-0.8,-1.2,-0.8,-2.2,5.5,
                          0.5,0.8,0.3,0.4,1.8,5.8,9.5,1.5,-1.2,0.5,
                          5.2,2.2,3.8,12.5],
    'CAB_pct_GDP':      [5.1,3.8,-2.5,-4.2,-3.8,-3.2,4.2,1.8,-4.2,-3.8,
                          9.2,1.5,-4.5,0.8,5.2,9.8,14.5,9.5,4.5,-8.2,
                          4.2,2.8,4.5,4.2,0.8,-3.2,-10.5,1.5,1.8,-2.5,
                          -15.5,-2.8,-4.5,-10.8],
    'Trade_Bal_USDbn':  [2.1,1.5,-0.8,-1.2,-1.0,-0.5,2.8,0.8,-1.8,-1.2,
                          5.8,0.5,-2.1,0.2,3.5,6.8,10.2,6.8,3.2,-4.5,
                          2.8,1.8,2.8,2.5,0.2,-1.8,-5.8,0.8,0.8,-1.2,
                          -8.5,-1.2,-2.2,-5.5],
    'OilRev_pct_TotalRev':[77.2,75.5,72.8,71.3,73.1,74.8,78.5,76.3,65.2,68.5,
                            83.2,80.5,75.8,78.5,81.2,85.5,88.2,84.5,82.8,68.5,
                            75.8,78.5,76.5,74.8,70.5,62.5,55.8,62.5,65.8,62.8,
                            55.5,60.5,62.8,58.8],
    'Oil_Price_USD':    [23.7,19.9,19.3,16.9,15.5,17.0,20.4,19.1,13.1,18.0,
                          28.5,25.0,25.0,28.9,38.3,53.4,65.1,72.4,99.6,61.8,
                          79.5,111.3,111.7,108.7,99.1,52.4,43.7,54.2,71.3,64.4,
                          41.7,70.9,99.0,82.5],
    'GDP_Growth_pct':   [8.2,3.1,2.3,2.0,0.9,-0.5,4.2,2.8,2.4,0.5,
                          5.3,4.4,3.8,7.4,6.6,6.5,6.1,6.8,6.3,6.9,
                          7.8,4.9,4.3,5.4,6.3,-1.6,-1.6,0.8,1.9,2.2,
                          -1.8,3.4,3.3,2.9],
    'Inflation_pct':    [7.5,13.0,44.5,57.2,57.0,72.8,29.3,8.5,10.0,6.6,
                          6.9,18.9,12.9,14.0,15.0,17.9,8.5,5.4,11.6,12.5,
                          13.7,10.8,12.2,8.5,8.1,9.0,15.7,16.5,12.1,11.4,
                          13.2,17.0,18.8,24.7],
    'M2_Growth_pct':    [45.8,38.2,55.8,48.7,35.0,28.5,22.8,18.5,15.2,22.5,
                          28.5,32.5,28.2,24.5,18.8,15.2,25.8,22.5,58.5,18.5,
                          14.5,18.5,15.8,12.8,14.5,16.8,18.5,12.5,22.5,18.5,
                          26.5,12.8,22.5,38.5],
    'Credit_Growth_pct':[28.3,24.5,38.5,32.1,28.5,22.3,18.5,15.2,12.8,22.5,
                          25.8,28.3,25.5,28.8,22.5,18.5,28.5,32.5,48.8,22.8,
                          18.5,22.5,15.5,14.8,16.5,18.5,22.8,14.5,18.5,15.8,
                          22.5,18.5,28.5,32.5],
    'TBill_Rate_pct':   [18.5,15.0,24.5,26.9,21.0,19.5,13.5,13.5,13.0,17.5,
                          12.0,14.5,18.9,15.0,14.2,13.0,10.0,9.5,10.5,18.5,
                          13.5,14.0,13.0,12.0,13.0,15.0,18.5,18.0,16.0,13.5,
                          12.0,11.5,18.5,22.0],
    'Real_IntRate_pct': [11.0,2.0,-20.0,-30.3,-36.0,-53.3,-15.8,5.0,3.0,11.0,
                          5.1,-4.4,6.0,1.0,-0.8,-4.9,1.5,4.1,-1.1,6.0,
                          -0.2,3.2,0.8,3.5,4.9,6.0,2.8,1.5,3.9,2.1,
                          -1.2,-5.5,-0.3,-2.7],
    'ExtDebt_pct_GDP':  [32.5,29.8,28.5,32.2,35.5,28.2,22.5,20.8,22.5,25.2,
                          22.8,21.5,20.2,18.5,14.5,8.2,5.5,4.5,5.8,8.5,
                          7.8,8.5,9.5,9.2,10.5,12.5,15.8,14.5,13.5,14.8,
                          18.5,20.5,22.5,26.5],
    'DSR_pct_Exports':  [12.8,11.5,14.2,15.8,17.2,16.5,12.3,10.5,11.8,12.5,
                          10.2,9.8,10.5,9.2,7.8,5.5,4.2,3.5,4.8,6.5,
                          5.2,5.8,5.5,5.2,5.8,7.2,8.5,7.8,6.8,7.2,
                          9.5,8.5,9.8,11.5],
    'ToT_Change_pct':   [4.2,-3.5,2.1,-5.8,-1.2,8.5,12.5,0.8,-25.5,35.2,
                          52.8,-6.5,-0.8,12.5,32.8,38.5,19.2,10.8,35.5,-38.5,
                          28.8,35.2,0.5,-0.8,-5.5,-48.5,-19.8,23.5,30.8,-0.8,
                          -33.8,63.5,-2.8,-17.2],
    'Currency_Crisis':  [0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,
                          0,0,0,0,0,1,1,0,0,0,1,1,1,1],
}

df = pd.DataFrame(raw)
print(f"Dataset shape      : {df.shape}")
print(f"Period             : {df['Year'].min()} – {df['Year'].max()}")
print(f"Currency Crisis    : {df['Currency_Crisis'].sum()} years  ({df['Currency_Crisis'].mean():.1%})")
print(f"No Crisis          : {(df['Currency_Crisis']==0).sum()} years  ({1-df['Currency_Crisis'].mean():.1%})")
df.head(10)


In [ ]:
# ── Quick info & missing-value check ─────────────────────────────────────────
print("\n📋 Dataset Info:")
df.info()
print("\n❓ Missing values:", df.isnull().sum().sum())
print("\n📊 Descriptive Statistics:")
df.describe().round(2)


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1 Exchange Rate & EMP Timeline ─────────────────────────────────────────
crisis_yrs = df[df['Currency_Crisis']==1]['Year'].values

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle("Nigeria: Key Currency Crisis Indicators (1990–2023)",
             fontsize=14, fontweight='bold')

def shade(ax):
    for yr in crisis_yrs:
        ax.axvspan(yr-0.5, yr+0.5, alpha=0.18, color=CRISIS_RED)

# Plot 1: Exchange rate & parallel premium
ax1t = axes[0].twinx()
axes[0].plot(df['Year'], df['ExRate_NGBUSD'],   color=NG_GREEN, lw=2.2, marker='o', ms=4, label='Official Rate (NGN/USD)')
ax1t.plot(df['Year'], df['Parallel_Premium'], color='#FF8F00', lw=1.8, ls='--', marker='s', ms=3.5, label='Parallel Premium (%)')
shade(axes[0])
axes[0].set_ylabel('NGN/USD', color=NG_GREEN, fontsize=9)
ax1t.set_ylabel('Parallel Premium (%)', color='#FF8F00', fontsize=9)
axes[0].set_title('Official Exchange Rate & Parallel Market Premium', fontweight='bold')
lines1, labs1 = axes[0].get_legend_handles_labels()
lines2, labs2 = ax1t.get_legend_handles_labels()
axes[0].legend(lines1+lines2, labs1+labs2, fontsize=8, loc='upper left')

# Plot 2: FX Reserves & EMP Index
ax2t = axes[1].twinx()
axes[1].fill_between(df['Year'], df['FX_Reserves_USDbn'], alpha=0.35, color=NG_GREEN)
axes[1].plot(df['Year'], df['FX_Reserves_USDbn'], color=NG_GREEN, lw=2, label='FX Reserves (USD Bn)')
ax2t.plot(df['Year'], df['EMP_Index'], color=CRISIS_RED, lw=1.8, marker='^', ms=4, label='EMP Index')
ax2t.axhline(df['EMP_Index'].mean() + 1.5*df['EMP_Index'].std(), color=CRISIS_RED, ls=':', lw=1.5, label='Crisis threshold')
shade(axes[1])
axes[1].set_ylabel('FX Reserves (USD Bn)', color=NG_GREEN, fontsize=9)
ax2t.set_ylabel('EMP Index', color=CRISIS_RED, fontsize=9)
axes[1].set_title('Foreign Reserves & Exchange Market Pressure Index', fontweight='bold')
l1,b1 = axes[1].get_legend_handles_labels()
l2,b2 = ax2t.get_legend_handles_labels()
axes[1].legend(l1+l2, b1+b2, fontsize=8)

# Plot 3: Oil price vs GDP growth
ax3t = axes[2].twinx()
axes[2].bar(df['Year'], df['Oil_Price_USD'], alpha=0.45, color=NG_GOLD, label='Oil Price (USD/bbl)')
ax3t.plot(df['Year'], df['GDP_Growth_pct'], color='#1565C0', lw=2, marker='D', ms=3.5, label='GDP Growth (%)')
ax3t.axhline(0, color='black', lw=0.8)
shade(axes[2])
axes[2].set_ylabel('Oil Price (USD/bbl)', color='#B8860B', fontsize=9)
ax3t.set_ylabel('GDP Growth (%)', color='#1565C0', fontsize=9)
axes[2].set_title('Oil Price & GDP Growth Rate', fontweight='bold')
axes[2].set_xlabel('Year', fontsize=10)
l1,b1 = axes[2].get_legend_handles_labels()
l2,b2 = ax3t.get_legend_handles_labels()
axes[2].legend(l1+l2, b1+b2, fontsize=8)

for ax in axes:
    ax.set_xlim(1989, 2024)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('nigeria_eda_timeline.png', bbox_inches='tight', dpi=130)
plt.show()
print("🔴 Red shading = currency crisis episodes")


In [ ]:
# ── 3.2 Correlation heatmap ───────────────────────────────────────────────────
feat_cols = [c for c in df.columns if c not in ['Year']]
corr = df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlGn_r', center=0,
            annot=True, fmt='.2f', annot_kws={'size':7},
            linewidths=0.35, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'shrink':0.7})
ax.set_title("Pearson Correlation Matrix — Nigeria Currency Crisis Variables",
             fontsize=12, fontweight='bold', pad=12)
plt.xticks(rotation=45, ha='right', fontsize=7.5)
plt.yticks(fontsize=7.5)
plt.tight_layout()
plt.savefig('nigeria_corr_heatmap.png', bbox_inches='tight', dpi=130)
plt.show()

print("\n📊 Top correlates with Currency_Crisis (absolute):")
print(corr['Currency_Crisis'].drop('Currency_Crisis').abs()
      .sort_values(ascending=False).round(3).to_string())


In [ ]:
# ── 3.3 KDE plots: crisis vs no-crisis ───────────────────────────────────────
key_vars = ['ExRate_Depr_pct','Parallel_Premium','Reserves_Change_pct',
            'Oil_Price_USD','EMP_Index','Inflation_pct']
labels   = ['Exch Rate Depreciation (%)','Parallel Market Premium (%)',
            'Reserves Change (%)','Oil Price (USD/bbl)',
            'EMP Index','Inflation (%)']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("Feature Distributions: Currency Crisis vs. No Crisis — Nigeria",
             fontsize=13, fontweight='bold')

for ax, var, lbl in zip(axes.flatten(), key_vars, labels):
    for val, color, name in [(0, NG_GREEN,'No Crisis'),(1, CRISIS_RED,'Currency Crisis')]:
        subset = df[df['Currency_Crisis']==val][var]
        subset.plot.kde(ax=ax, color=color, lw=2.2, label=name)
        ax.axvline(subset.mean(), color=color, ls='--', lw=1.2, alpha=0.7)
    ax.set_title(lbl, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('nigeria_kde_plots.png', bbox_inches='tight', dpi=130)
plt.show()


## 4. EMP Index Construction & Target Label Verification

In [ ]:
# ── 4.1 Construct EMP index from components ───────────────────────────────────
# EMP_t = w1*(ΔE/E) − w2*(ΔR/R)
# Weights = inverse of standard deviation (precision weights)

delta_e = df['ExRate_Depr_pct'].values        # % change in exchange rate
delta_r = df['Reserves_Change_pct'].values    # % change in reserves

w1 = 1 / delta_e.std()
w2 = 1 / delta_r.std()

emp_constructed = w1 * delta_e - w2 * delta_r
emp_mean   = emp_constructed.mean()
emp_std    = emp_constructed.std()
threshold  = emp_mean + 1.5 * emp_std

crisis_emp = (emp_constructed > threshold).astype(int)

print(f"EMP Statistics:")
print(f"  Mean    : {emp_mean:.4f}")
print(f"  Std Dev : {emp_std:.4f}")
print(f"  Crisis threshold (μ + 1.5σ) : {threshold:.4f}")
print(f"\nCrisis years by EMP rule    : {crisis_emp.sum()}")
print(f"Crisis years in dataset     : {df['Currency_Crisis'].sum()}")

# Also flag by composite rule: |ΔE/E| > 15% OR ΔR/R < -15%
crisis_composite = ((np.abs(delta_e) > 15) | (delta_r < -15)).astype(int)
print(f"Crisis years by composite   : {crisis_composite.sum()}")
print(f"\nFinal target (dataset)  — Crisis years: {df['Currency_Crisis'].sum()} | Non-crisis: {(df['Currency_Crisis']==0).sum()}")


In [ ]:
# ── 4.2 Visualise EMP and crisis episodes ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# EMP constructed vs threshold
axes[0].plot(df['Year'], emp_constructed, color=NG_GREEN, lw=2, marker='o', ms=4.5, label='EMP Index (constructed)')
axes[0].axhline(threshold, color=CRISIS_RED, ls='--', lw=1.8, label=f'Crisis threshold ({threshold:.2f})')
axes[0].axhline(emp_mean,  color='grey', ls=':',  lw=1.2, label=f'Mean ({emp_mean:.2f})')
for yr in crisis_yrs:
    axes[0].axvspan(yr-0.4, yr+0.4, alpha=0.2, color=CRISIS_RED)
axes[0].fill_between(df['Year'], emp_constructed, threshold,
                     where=emp_constructed > threshold, alpha=0.3, color=CRISIS_RED)
axes[0].set_ylabel('EMP Value', fontsize=10)
axes[0].set_title('Exchange Market Pressure Index — Nigeria (1990–2023)', fontweight='bold')
axes[0].legend(fontsize=9)

# Binary crisis indicator
axes[1].bar(df['Year'], df['Currency_Crisis'],
            color=[CRISIS_RED if v else NG_GREEN for v in df['Currency_Crisis']],
            alpha=0.8, width=0.75)
axes[1].set_ylabel('Crisis (1) / No Crisis (0)', fontsize=10)
axes[1].set_xlabel('Year', fontsize=10)
axes[1].set_title('Binary Currency Crisis Indicator', fontweight='bold')
axes[1].set_xlim(1989, 2024)
axes[1].tick_params(axis='x', rotation=45)

red_p  = mpatches.Patch(color=CRISIS_RED, label='Currency Crisis')
grn_p  = mpatches.Patch(color=NG_GREEN,   label='No Crisis')
axes[1].legend(handles=[red_p, grn_p], fontsize=9)

plt.tight_layout()
plt.savefig('nigeria_emp_crisis.png', bbox_inches='tight', dpi=130)
plt.show()


## 5. Feature Engineering & Preprocessing

In [ ]:
# ── 5.1 Engineer additional features ─────────────────────────────────────────
df_fe = df.copy()

# Lag features (t-1): EWS needs leading indicators
lag_vars = ['ExRate_Depr_pct','Parallel_Premium','FX_Reserves_USDbn',
            'Reserves_Change_pct','EMP_Index','Oil_Price_USD','Inflation_pct',
            'M2_Growth_pct']
for col in lag_vars:
    df_fe[f'{col}_lag1'] = df_fe[col].shift(1)
    df_fe[f'{col}_lag2'] = df_fe[col].shift(2)

# Oil vulnerability score (combined oil dependence + oil price signal)
df_fe['Oil_Vulnerability'] = (df_fe['OilRev_pct_TotalRev'] / 100) * (1 / (df_fe['Oil_Price_USD'] + 1e-6))

# Import cover ratio (reserves in months of imports proxy)
df_fe['Import_Cover_Proxy'] = df_fe['FX_Reserves_USDbn'] / (np.abs(df_fe['Trade_Bal_USDbn']) + 0.1)

# Currency over-valuation proxy (real rate signal)
df_fe['RER_Misalignment'] = df_fe['Inflation_pct'] - df_fe['ExRate_Depr_pct']

# Monetary pressure index
df_fe['MonPressure'] = df_fe['M2_Growth_pct'] - df_fe['GDP_Growth_pct']

# EMP momentum
df_fe['EMP_Change'] = df_fe['EMP_Index'].diff()

# Reserve adequacy gap (below 3-month rule)
df_fe['Reserves_Gap'] = 3.0 - df_fe['Import_Cover_Proxy']

# Drop NaN rows from lag operations
df_fe.dropna(inplace=True)
df_fe.reset_index(drop=True, inplace=True)

print(f"Dataset after feature engineering: {df_fe.shape}")
print(f"Total features available       : {df_fe.shape[1] - 2}  (excl. Year & target)")


In [ ]:
# ── 5.2 Define X and y ────────────────────────────────────────────────────────
feature_cols = [c for c in df_fe.columns if c not in ['Year','Currency_Crisis']]

X = df_fe[feature_cols].values
y = df_fe['Currency_Crisis'].values

print(f"Feature matrix X : {X.shape}")
print(f"Target vector  y : {y.shape}  →  Crisis: {y.sum()} | No-Crisis: {(y==0).sum()}")
print(f"\nAll {len(feature_cols)} features:")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {f}")


In [ ]:
# ── 5.3 Scale features ────────────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── 5.4 SMOTE to handle class imbalance ──────────────────────────────────────
smote    = SMOTE(random_state=42, k_neighbors=3)
X_res, y_res = smote.fit_resample(X_scaled, y)
print(f"After SMOTE resampling:  Crisis={y_res.sum()}  |  No-Crisis={(y_res==0).sum()}")

# ── 5.5 Class weight ratio (alternative to SMOTE) ────────────────────────────
ratio = (y == 0).sum() / (y == 1).sum()
print(f"Class weight ratio for class_weight param: {ratio:.2f}")


## 6. Random Forest Training & Hyperparameter Tuning

In [ ]:
# ── 6.1 Baseline Random Forest ────────────────────────────────────────────────
rf_base = RandomForestClassifier(
    n_estimators    = 300,
    max_depth       = None,
    max_features    = 'sqrt',
    min_samples_leaf= 1,
    class_weight    = 'balanced',
    oob_score       = True,
    random_state    = 42,
    n_jobs          = -1,
)
rf_base.fit(X_scaled, y)
y_pred_base  = rf_base.predict(X_scaled)
y_proba_base = rf_base.predict_proba(X_scaled)[:, 1]

print("Baseline Random Forest — Training Performance")
print("=" * 48)
print(f"  OOB Score  : {rf_base.oob_score_:.4f}")
print(f"  AUC-ROC    : {roc_auc_score(y, y_proba_base):.4f}")
print(f"  F1-Score   : {f1_score(y, y_pred_base):.4f}")
print(f"  Accuracy   : {accuracy_score(y, y_pred_base):.4f}")
print("\nNote: High in-sample scores expected; CV scores are more meaningful.")


In [ ]:
# ── 6.2 Bayesian Hyperparameter Optimisation (Optuna) ────────────────────────
tscv = TimeSeriesSplit(n_splits=5)

def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators',   100, 600),
        'max_depth'       : trial.suggest_int('max_depth',        3,  20),
        'max_features'    : trial.suggest_categorical('max_features', ['sqrt','log2', 0.5, 0.7]),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1,   8),
        'min_samples_split': trial.suggest_int('min_samples_split',2, 10),
        'class_weight'    : 'balanced',
        'oob_score'       : False,
        'random_state'    : 42,
        'n_jobs'          : -1,
    }
    model  = RandomForestClassifier(**params)
    scores = cross_val_score(model, X_scaled, y, cv=tscv,
                             scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=False)

print(f"✅  Best AUC-ROC (CV) : {study.best_value:.4f}")
print(f"\nBest hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k:<22} : {v}")


In [ ]:
# ── 6.3 Train final model with best parameters ───────────────────────────────
best_params = study.best_params.copy()
best_params.update({ 'class_weight':'balanced', 'oob_score':True, 'random_state':42, 'n_jobs':-1 })

rf_final = RandomForestClassifier(**best_params)
rf_final.fit(X_scaled, y)

y_pred  = rf_final.predict(X_scaled)
y_proba = rf_final.predict_proba(X_scaled)[:, 1]

print("Final Random Forest — Training Report")
print("=" * 48)
print(f"  OOB Score  : {rf_final.oob_score_:.4f}")
print()
print(classification_report(y, y_pred, target_names=['No Crisis','Currency Crisis']))


## 7. Model Evaluation

In [ ]:
# ── 7.1 Walk-Forward CV comparison across algorithms ────────────────────────
print("🔄  Walk-Forward Cross-Validation (TimeSeriesSplit, n=5)")
print("=" * 58)

models = {
    'Random Forest (Tuned)' : rf_final,
    'Gradient Boosting'     : GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                max_depth=3, random_state=42),
    'Logistic Regression'   : LogisticRegression(class_weight='balanced', max_iter=500, random_state=42),
    'Decision Tree'         : DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=42),
    'K-Nearest Neighbours'  : KNeighborsClassifier(n_neighbors=5),
}

results = {}
for name, model in models.items():
    auc  = cross_val_score(model, X_scaled, y, cv=tscv, scoring='roc_auc').mean()
    f1   = cross_val_score(model, X_scaled, y, cv=tscv, scoring='f1').mean()
    rec  = cross_val_score(model, X_scaled, y, cv=tscv, scoring='recall').mean()
    prec = cross_val_score(model, X_scaled, y, cv=tscv, scoring='precision').mean()
    results[name] = {'AUC-ROC':auc, 'F1-Score':f1, 'Recall':rec, 'Precision':prec}
    print(f"  {name:<26} AUC={auc:.3f}  F1={f1:.3f}  Recall={rec:.3f}  Prec={prec:.3f}")

results_df = pd.DataFrame(results).T.round(3)
print()
print(results_df.to_string())


In [ ]:
# ── 7.2 Confusion Matrix + ROC Curve + Precision-Recall ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Crisis','Crisis'],
            yticklabels=['No Crisis','Crisis'],
            ax=axes[0], linewidths=1.2, linecolor='white',
            annot_kws={'size':13})
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=12)
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')

# 2. ROC Curve
RocCurveDisplay.from_predictions(y, y_proba, ax=axes[1],
    color=NG_GREEN, name=f'Random Forest (AUC={roc_auc_score(y, y_proba):.3f})')
axes[1].plot([0,1],[0,1],'k--', lw=1, alpha=0.6, label='Random Classifier')
axes[1].set_title('ROC Curve — Currency Crisis EWS', fontweight='bold', fontsize=12)
axes[1].legend(fontsize=9)

# 3. Precision-Recall curve
prec_vals, rec_vals, thresholds = precision_recall_curve(y, y_proba)
axes[2].plot(rec_vals, prec_vals, color=CRISIS_RED, lw=2.2, label='RF Precision-Recall')
axes[2].fill_between(rec_vals, prec_vals, alpha=0.12, color=CRISIS_RED)
axes[2].axhline(y.mean(), color='grey', ls='--', lw=1.2, label=f'Baseline ({y.mean():.2f})')
axes[2].set_xlabel('Recall', fontsize=10); axes[2].set_ylabel('Precision', fontsize=10)
axes[2].set_title('Precision-Recall Curve', fontweight='bold', fontsize=12)
axes[2].legend(fontsize=9)
axes[2].set_xlim(0,1); axes[2].set_ylim(0,1.05)

plt.suptitle("Random Forest — Nigeria Currency Crisis Model Evaluation", fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('nigeria_model_evaluation.png', bbox_inches='tight', dpi=130)
plt.show()

print(f"\n📈 Key Metrics (Full Training Set):")
print(f"  OOB Score  : {rf_final.oob_score_:.4f}")
print(f"  AUC-ROC    : {roc_auc_score(y, y_proba):.4f}")
print(f"  F1-Score   : {f1_score(y, y_pred):.4f}")
print(f"  Recall     : {recall_score(y, y_pred):.4f}")
print(f"  Precision  : {precision_score(y, y_pred):.4f}")
print(f"  Brier Score: {brier_score_loss(y, y_proba):.4f}")


In [ ]:
# ── 7.3 Model comparison bar chart ───────────────────────────────────────────
metric_names = ['AUC-ROC','F1-Score','Recall','Precision']
x = np.arange(len(metric_names))
width = 0.16
colors = [NG_GREEN,'#1565C0','#FF8F00','#9C27B0','#E53935']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (name, row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in metric_names]
    ax.bar(x + i*width, vals, width, label=name, color=colors[i], alpha=0.85)

ax.set_xticks(x + width*2)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Comparison — Walk-Forward CV (5-Fold, Time-Series Aware)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8.5, loc='lower right', ncol=2)
ax.axhline(0.85, color='grey', ls=':', lw=1.2, label='Target AUC')
plt.tight_layout()
plt.savefig('nigeria_model_comparison.png', bbox_inches='tight', dpi=130)
plt.show()


## 8. SHAP Explainability & Feature Importance

In [ ]:
# ── 8.1 Compute SHAP values ───────────────────────────────────────────────────
explainer   = shap.TreeExplainer(rf_final)
shap_values = explainer.shap_values(X_scaled)

# shap_values is a list [class_0, class_1]; use class 1 (crisis)
sv_crisis = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_df   = pd.DataFrame(np.abs(sv_crisis), columns=feature_cols)
mean_shap = shap_df.mean().sort_values(ascending=False)

print("📊 SHAP Feature Importance Ranking (Mean |SHAP Value| — Crisis class)")
print("=" * 62)
for i, (feat, val) in enumerate(mean_shap.head(20).items(), 1):
    bar = '█' * int(val * 120)
    print(f"  {i:2d}. {feat:<40} {val:.5f}  {bar}")


In [ ]:
# ── 8.2 SHAP Summary (Beeswarm) Plot ─────────────────────────────────────────
plt.figure(figsize=(11, 9))
shap.summary_plot(sv_crisis, X_scaled, feature_names=feature_cols,
                  show=False, max_display=18,
                  color_bar_label='Feature Value (Low → High)')
plt.title("SHAP Summary Plot — Random Forest Currency Crisis Model (Nigeria)",
          fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('nigeria_shap_summary.png', bbox_inches='tight', dpi=130)
plt.show()
print("\n💡 Red = high feature value;  Blue = low feature value")
print("   Positive SHAP → pushes prediction toward CRISIS")
print("   Negative SHAP → pushes prediction toward NO CRISIS")


In [ ]:
# ── 8.3 Top-15 SHAP Bar Chart ─────────────────────────────────────────────────
top_n  = 15
top_ft = mean_shap.head(top_n)
colors_bar = [CRISIS_RED if i < 4 else NG_GOLD if i < 8 else NG_GREEN for i in range(top_n)]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(top_n), top_ft.values[::-1], color=colors_bar[::-1], alpha=0.88)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_ft.index[::-1], fontsize=9.5)
ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax.set_title('Top-15 Feature Importance (SHAP) — Nigeria Currency Crisis EWS',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, top_ft.values[::-1]):
    ax.text(val+0.0002, bar.get_y()+bar.get_height()/2,
            f'{val:.5f}', va='center', fontsize=8.5)
red_p  = mpatches.Patch(color=CRISIS_RED, label='Critical drivers')
gold_p = mpatches.Patch(color=NG_GOLD,    label='Moderate drivers')
grn_p  = mpatches.Patch(color=NG_GREEN,   label='Supporting drivers')
ax.legend(handles=[red_p, gold_p, grn_p], fontsize=9)
plt.tight_layout()
plt.savefig('nigeria_shap_bar.png', bbox_inches='tight', dpi=130)
plt.show()


In [ ]:
# ── 8.4 SHAP Waterfall: 2023 (most recent crisis year) ───────────────────────
idx_2023 = df_fe[df_fe['Year']==2023].index[0]
print(f"SHAP Waterfall — Nigeria 2023")
print(f"  Actual label        : {'CRISIS' if y[idx_2023] else 'No Crisis'}")
print(f"  Crisis probability  : {y_proba[idx_2023]:.4f}")

shap.waterfall_plot(
    shap.Explanation(values=sv_crisis[idx_2023],
                     base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
                     data=X_scaled[idx_2023],
                     feature_names=feature_cols),
    max_display=15, show=False)
plt.title("SHAP Waterfall — Nigeria 2023 (FX Unification Crisis)", fontsize=11, pad=25)
plt.tight_layout()
plt.savefig('nigeria_shap_waterfall_2023.png', bbox_inches='tight', dpi=130)
plt.show()


In [ ]:
# ── 8.5 MDI Feature Importance vs SHAP comparison ────────────────────────────
mdi_importance = pd.Series(rf_final.feature_importances_, index=feature_cols)
mdi_top = mdi_importance.sort_values(ascending=False).head(15)
shap_top = mean_shap.head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# MDI
axes[0].barh(range(len(mdi_top)), mdi_top.values[::-1], color=NG_GREEN, alpha=0.85)
axes[0].set_yticks(range(len(mdi_top))); axes[0].set_yticklabels(mdi_top.index[::-1], fontsize=9)
axes[0].set_xlabel('Mean Decrease in Impurity', fontsize=10)
axes[0].set_title('MDI Feature Importance (RF built-in)', fontweight='bold')
# SHAP
axes[1].barh(range(len(shap_top)), shap_top.values[::-1], color=CRISIS_RED, alpha=0.85)
axes[1].set_yticks(range(len(shap_top))); axes[1].set_yticklabels(shap_top.index[::-1], fontsize=9)
axes[1].set_xlabel('Mean |SHAP Value|', fontsize=10)
axes[1].set_title('SHAP Feature Importance (Model-agnostic)', fontweight='bold')
plt.suptitle("MDI vs SHAP Feature Importance — Nigeria Currency Crisis RF", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('nigeria_mdi_vs_shap.png', bbox_inches='tight', dpi=130)
plt.show()


## 9. Walk-Forward Cross-Validation (Detailed Fold Analysis)

In [ ]:
# ── 9.1 Fold-by-fold performance ─────────────────────────────────────────────
tscv5 = TimeSeriesSplit(n_splits=5)
fold_results = []

print("Walk-Forward CV — Fold-Level Performance (Random Forest)")
print("=" * 65)
print(f"{'Fold':>5} | {'Train Size':>10} | {'Test Size':>9} | {'AUC-ROC':>8} | {'F1':>7} | {'Recall':>7} | {'Prec':>7}")
print("-" * 65)

for fold, (train_idx, test_idx) in enumerate(tscv5.split(X_scaled, y), 1):
    X_tr, X_te = X_scaled[train_idx], X_scaled[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    # Check if test set has both classes
    if len(np.unique(y_te)) < 2:
        print(f"  {fold:>3}  | {len(train_idx):>10} | {len(test_idx):>9} | {'N/A (single class in test)':>40}")
        continue

    rf_cv = RandomForestClassifier(**best_params)
    rf_cv.fit(X_tr, y_tr)
    yp_te  = rf_cv.predict(X_te)
    ypr_te = rf_cv.predict_proba(X_te)[:, 1]

    auc  = roc_auc_score(y_te, ypr_te)
    f1v  = f1_score(y_te, yp_te, zero_division=0)
    recv = recall_score(y_te, yp_te, zero_division=0)
    prv  = precision_score(y_te, yp_te, zero_division=0)
    fold_results.append({'Fold':fold,'AUC':auc,'F1':f1v,'Recall':recv,'Prec':prv})
    yr_range = f"{df_fe['Year'].values[train_idx[0]]}–{df_fe['Year'].values[train_idx[-1]]}"
    print(f"  {fold:>3}  | {yr_range:>10} | {len(test_idx):>9} | {auc:>8.4f} | {f1v:>7.4f} | {recv:>7.4f} | {prv:>7.4f}")

fr_df = pd.DataFrame(fold_results)
if len(fr_df):
    print("-" * 65)
    print(f"{'Mean':>5} | {'':>10} | {'':>9} | {fr_df['AUC'].mean():>8.4f} | {fr_df['F1'].mean():>7.4f} | {fr_df['Recall'].mean():>7.4f} | {fr_df['Prec'].mean():>7.4f}")
    print(f"{'Std':>5} | {'':>10} | {'':>9} | {fr_df['AUC'].std():>8.4f} | {fr_df['F1'].std():>7.4f} | {fr_df['Recall'].std():>7.4f} | {fr_df['Prec'].std():>7.4f}")


## 10. Early Warning Score (EWS) & Policy Signals

In [ ]:
# ── 10.1 Generate EWS probability scores ─────────────────────────────────────
df_ews = df_fe[['Year','Currency_Crisis']].copy()
df_ews['Crisis_Probability'] = y_proba
df_ews['Predicted']          = y_pred

# Alert levels
df_ews['Alert_Level'] = pd.cut(
    df_ews['Crisis_Probability'],
    bins   = [0, 0.30, 0.55, 0.75, 1.0],
    labels = ['🟢 Low', '🟡 Moderate', '🟠 High', '🔴 Critical']
)

print("📊 Nigeria Currency Crisis — EWS Probability Scores (1992–2023)")
print("=" * 72)
print(f"{'Year':>6}  {'Actual':>12}  {'Prob':>8}  {'Predicted':>12}  {'Alert':>14}")
print("-" * 72)
for _, row in df_ews.iterrows():
    actual = "CRISIS" if row['Currency_Crisis'] else "No Crisis"
    pred   = "CRISIS" if row['Predicted']       else "No Crisis"
    print(f"{int(row['Year']):>6}  {actual:>12}  {row['Crisis_Probability']:>8.4f}  {pred:>12}  {str(row['Alert_Level']):>16}")


In [ ]:
# ── 10.2 EWS Probability Timeline ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6.5))

# Colour bands
ax.axhspan(0.75, 1.05, alpha=0.08, color=CRISIS_RED, label='Critical zone (>0.75)')
ax.axhspan(0.55, 0.75, alpha=0.08, color='#FF8F00')
ax.axhspan(0.30, 0.55, alpha=0.06, color=NG_GOLD)

ax.fill_between(df_ews['Year'], df_ews['Crisis_Probability'],
                where=df_ews['Crisis_Probability'] >= 0.55,
                alpha=0.35, color=CRISIS_RED)
ax.fill_between(df_ews['Year'], df_ews['Crisis_Probability'],
                where=df_ews['Crisis_Probability'] < 0.55,
                alpha=0.25, color=NG_GREEN)
ax.plot(df_ews['Year'], df_ews['Crisis_Probability'],
        color='#004D00', lw=2.2, marker='o', ms=5.5, label='Crisis Probability')

# Threshold lines
ax.axhline(0.55, color=CRISIS_RED, ls='--', lw=1.8, label='Alert threshold (0.55)')
ax.axhline(0.75, color='#8B0000', ls='--', lw=1.5, label='Critical threshold (0.75)')
ax.axhline(0.30, color=NG_GREEN,  ls=':',  lw=1.2, label='Low-risk threshold (0.30)')

# Shade actual crisis years
for yr in df_ews[df_ews['Currency_Crisis']==1]['Year']:
    ax.axvspan(yr-0.4, yr+0.4, alpha=0.10, color=CRISIS_RED)

# Annotate major episodes
annotations = {2016:"2016
Naira Float", 2023:"2023
FX Unification", 2009:"2009
GFC"}
for yr, lbl in annotations.items():
    if yr in df_ews['Year'].values:
        prob = df_ews[df_ews['Year']==yr]['Crisis_Probability'].values[0]
        ax.annotate(lbl, xy=(yr, prob), xytext=(yr, prob+0.12),
                    fontsize=8, color=CRISIS_RED, ha='center',
                    arrowprops=dict(arrowstyle='->', color=CRISIS_RED, lw=1.2))

ax.set_xlim(df_ews['Year'].min()-0.5, df_ews['Year'].max()+0.5)
ax.set_ylim(-0.05, 1.1)
ax.set_xlabel('Year', fontsize=11); ax.set_ylabel('Currency Crisis Probability', fontsize=11)
ax.set_title('Nigeria Currency Crisis — Early Warning Score (Random Forest, 1992–2023)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=8.5, loc='upper left', ncol=2)
ax.set_xticks(df_ews['Year'][::2]); ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('nigeria_ews_timeline.png', bbox_inches='tight', dpi=130)
plt.show()
print("\n🚨 Crisis probability > 0.55 → CBN Emergency FX Reserve Review Protocol")


In [ ]:
# ── 10.3 Optimal Threshold Analysis ──────────────────────────────────────────
prec_vals, rec_vals, thresh_vals = precision_recall_curve(y, y_proba)
f1_per_thresh = 2 * prec_vals * rec_vals / (prec_vals + rec_vals + 1e-9)
opt_idx   = np.argmax(f1_per_thresh)
opt_thresh = thresh_vals[opt_idx] if opt_idx < len(thresh_vals) else 0.5

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_vals, prec_vals[:-1], label='Precision', color=NG_GREEN,  lw=2)
ax.plot(thresh_vals, rec_vals[:-1],  label='Recall',    color=CRISIS_RED, lw=2)
ax.plot(thresh_vals, f1_per_thresh[:-1], label='F1-Score', color=NG_GOLD, lw=2, ls='--')
ax.axvline(opt_thresh, color='black', ls=':', lw=1.5, label=f'Optimal threshold ({opt_thresh:.2f})')
ax.axvline(0.55, color='#FF8F00', ls='--', lw=1.3, label='Policy threshold (0.55)')
ax.set_xlabel('Classification Threshold', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Threshold Optimisation — Nigeria Currency Crisis EWS', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1.05)
plt.tight_layout()
plt.savefig('nigeria_threshold_analysis.png', bbox_inches='tight', dpi=130)
plt.show()
print(f"\nOptimal F1 threshold : {opt_thresh:.4f}")
print(f"At optimal threshold — Precision: {prec_vals[opt_idx]:.4f} | Recall: {rec_vals[opt_idx]:.4f} | F1: {f1_per_thresh[opt_idx]:.4f}")


## 11. Summary Report

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
cv_auc  = cross_val_score(rf_final, X_scaled, y, cv=tscv, scoring='roc_auc').mean()
cv_f1   = cross_val_score(rf_final, X_scaled, y, cv=tscv, scoring='f1').mean()
cv_rec  = cross_val_score(rf_final, X_scaled, y, cv=tscv, scoring='recall').mean()
cv_prec = cross_val_score(rf_final, X_scaled, y, cv=tscv, scoring='precision').mean()

print("=" * 68)
print("  NIGERIA CURRENCY CRISIS EARLY WARNING MODEL — SUMMARY REPORT")
print("=" * 68)
print(f"\n  DATASET")
print(f"    Period             : 1990–2023 (annual, CBN/IMF/WB)")
print(f"    Observations       : {len(df_fe)} (after lag feature engineering)")
print(f"    Currency Crisis    : {y.sum()} years ({y.mean():.1%})  |  No Crisis: {(y==0).sum()} years")
print(f"    Total features     : {len(feature_cols)}")
print(f"\n  ALGORITHM")
print(f"    Primary model      : Random Forest (Bootstrap Aggregating)")
print(f"    n_estimators       : {best_params.get('n_estimators')}")
print(f"    max_depth          : {best_params.get('max_depth')}")
print(f"    OOB Score          : {rf_final.oob_score_:.4f}")
print(f"    Hyperopt           : Bayesian (Optuna, 100 trials)")
print(f"    Validation         : Walk-forward 5-fold CV (time-aware)")
print(f"\n  CROSS-VALIDATED PERFORMANCE")
print(f"    AUC-ROC            : {cv_auc:.4f}")
print(f"    F1-Score           : {cv_f1:.4f}")
print(f"    Recall             : {cv_rec:.4f}")
print(f"    Precision          : {cv_prec:.4f}")
print(f"\n  TOP-5 CRISIS DRIVERS (SHAP)")
for i, (feat, val) in enumerate(mean_shap.head(5).items(), 1):
    print(f"    {i}. {feat:<40} SHAP={val:.5f}")
print(f"\n  CBN POLICY RECOMMENDATIONS")
recs = [
    "Maintain FX reserves ≥ 6 months import cover (primary crisis buffer)",
    "Parallel market premium > 20% → activate FX market unification measures",
    "Oil revenue > 70% of total revenue → mandatory savings/SWF rule",
    "M2 growth > 25% p.a. without commensurate GDP growth → tighten CBN policy",
    "EWS probability > 0.55 → CBN emergency FX committee review activation",
]
for i, r in enumerate(recs, 1):
    print(f"    {i}. {r}")
print("\n" + "=" * 68)


---
## 📚 Data Sources
- **Central Bank of Nigeria (CBN) Statistical Bulletin** — Exchange rates, reserves, M2, credit
- **IMF International Financial Statistics (IFS)** — BOP, EMP components, external debt
- **World Bank World Development Indicators (WDI)** — GDP growth, inflation, trade balance
- **OPEC Annual Statistical Bulletin** — Oil prices, Nigerian crude production
- **IMF World Economic Outlook (WEO)** — Fiscal aggregates, current account

## 📖 Key References
- Kaminsky, G., Lizondo, S. & Reinhart, C.M. (1998). *Leading Indicators of Currency Crises*. IMF Staff Papers.
- Rose, A.K. & Spiegel, M.M. (2012). *Cross-country causes and consequences of the crisis*. IMF Economic Review.
- Peltonen, T.A. (2006). *Are Emerging Market Currency Crises Predictable? — A Test*. ECB Working Paper.
- Frankel, J. & Saravelos, G. (2012). *Can leading indicators assess country vulnerability?* Journal of International Economics.

## ⚖️ Usage
For academic and research purposes. Cite primary data sources when publishing results.
